# Coffee17: GAP vs GAP + Auxiliary Hierarchy

Screening validation seed 42 untuk mengisolasi efek hierarchy tanpa HBP. H0 memakai mapping pair+singleton lama; H0f memakai tujuh keluarga penuh. Bobot auxiliary tetap 0.2.

In [ ]:
# 1/5 SETUP REPO, DRIVE, DAN DATASET BERSIH
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
REPO = Path('/content/coffee-bean-classification')
DATA = Path('/content/coffee17-gap-hierarchy-data')
ORIGINAL = DATA / 'original'
CLEAN = DATA / 'clean'
FOLD = CLEAN / 'folds' / 'fold_1'
OUTPUT = Path('/content/drive/MyDrive/coffee17-gap-hierarchy')

if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ediprin/coffee-bean-classification.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)

if not (ORIGINAL / 'source' / 'train').is_dir():
    subprocess.run([
        sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_coffee17',
        '--output', str(ORIGINAL), '--archive', str(DATA / 'coffee17.zip'), '--seed', '42',
    ], check=True)
subprocess.run([
    sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_clean_grouped_folds',
    '--source-root', str(ORIGINAL / 'source'), '--output-root', str(CLEAN),
    '--expected-count', '979', '--folds', '5', '--seed', '42', '--validation-ratio', '0.1',
], check=True)
assert (FOLD / 'source' / 'val').is_dir(), f'Dataset belum siap: {FOLD}'
import torch
assert torch.cuda.is_available(), 'Aktifkan GPU T4 pada Runtime > Change runtime type'
print('GPU    :', torch.cuda.get_device_name(0))
print('DATASET:', FOLD)
print('OUTPUT :', OUTPUT)

In [ ]:
# 2/5 SCREENING VALIDATION SEED 42: M0 vs H0 (TANPA HBP)
cmd = [
    sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_finegrained_screening',
    '--data-root', str(FOLD), '--output-root', str(OUTPUT),
    '--stage', 'gap_hierarchy', '--seeds', '42', '--evaluation-split', 'val',
]
print('MENJALANKAN:', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
# 3/5 TAMPILKAN HASIL MAPPING PAIR+SINGLETON H0
import json
report = OUTPUT / 'val_reports' / 'M0_vs_H0_aggregate.json'
assert report.is_file(), f'Report belum ditemukan: {report}'
result = json.loads(report.read_text())
print('=== M0 GAP vs H0 GAP + AUXILIARY HIERARCHY ===')
for key, label in [('macro_f1', 'Macro-F1'), ('hard_class_f1', 'Hard-F1'), ('worst_class_f1', 'Worst-F1')]:
    row = result['summary'][key]
    print(f"{label:9s}: {row['baseline_mean']:.2%} -> {row['candidate_mean']:.2%} ({row['delta_mean']:+.2%})")
print('\nLolos screening hanya jika Macro-F1 dan Hard-F1 naik, serta Worst-F1 tidak turun >1 poin.')
print('H0 adalah diagnosis mapping lama; jangan buka test.')

In [ ]:
# 4/5 SCREENING H0f: TUJUH KELUARGA PENUH, BOBOT TETAP 0.2
# M0 akan dilewati otomatis bila training/evaluasinya sudah lengkap.
full_cmd = [
    sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_finegrained_screening',
    '--data-root', str(FOLD), '--output-root', str(OUTPUT),
    '--stage', 'gap_hierarchy_full', '--seeds', '42', '--evaluation-split', 'val',
]
print('MENJALANKAN:', ' '.join(full_cmd), flush=True)
subprocess.run(full_cmd, check=True)

In [ ]:
# 5/5 TAMPILKAN PUTUSAN H0f
full_report = OUTPUT / 'val_reports' / 'M0_vs_H0f_aggregate.json'
assert full_report.is_file(), f'Report belum ditemukan: {full_report}'
full_result = json.loads(full_report.read_text())
print('=== M0 GAP vs H0f GAP + FULL-FAMILY HIERARCHY ===')
passed = True
for key, label in [('macro_f1', 'Macro-F1'), ('hard_class_f1', 'Hard-F1'), ('worst_class_f1', 'Worst-F1')]:
    row = full_result['summary'][key]
    print(f"{label:9s}: {row['baseline_mean']:.2%} -> {row['candidate_mean']:.2%} ({row['delta_mean']:+.2%})")
    if key in ('macro_f1', 'hard_class_f1') and row['delta_mean'] <= 0:
        passed = False
    if key == 'worst_class_f1' and row['delta_mean'] < -0.01:
        passed = False
print('KEPUTUSAN:', 'PASS' if passed else 'FAIL')
print('Jika FAIL: hierarchy dihentikan. Jika PASS: mapping dibekukan sebelum seed lain/test.')